# 02 — All-in-One SGLang Self-Hosting & Benchmark on Kaggle (NVIDIA T4)

This notebook is **100% self-contained** — you only need to upload this `.ipynb` file to Kaggle, click **Run All**, and take the formatted results table at the bottom.

**Important Kaggle Setup:** In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x2** (not P100) and turn **Internet ON**.


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q "sglang[all]"


In [ ]:
"""Prompt pools and multi-turn agent conversation definitions for vLLM vs SGLang."""

from __future__ import annotations

SINGLE_REQUEST_PROMPT = "Explain concurrent users in two sentences."

CONCURRENT_PROMPT_POOL: list[str] = [
    "Explain how TCP three-way handshakes establish reliable connections.",
    "Describe the difference between processes and threads in Linux.",
    "Summarize how B-trees optimize database indexing on disk.",
    "What is the role of the page table in virtual memory management?",
    "Explain how HTTPS TLS session resumption reduces handshake latency.",
    "Compare optimistic locking and pessimistic locking in distributed databases.",
    "How does garbage collection mark-and-sweep avoid memory leaks?",
    "Explain the CAP theorem with a real-world distributed storage example.",
    "What happens inside a CPU pipeline during a branch misprediction?",
    "Describe how consistent hashing minimizes key redistribution in caches.",
    "Explain the difference between L1, L2, and L3 CPU caches.",
    "How does Raft consensus elect a new leader after a network partition?",
    "What is write-ahead logging (WAL) and why do databases rely on it?",
    "Explain how epoll scales better than select for thousands of sockets.",
    "Compare row-oriented and column-oriented data warehouse storage formats.",
    "How does bloom filtering save disk lookups in LSM-tree storage engines?",
    "Explain vector clock causality tracking in leaderless replication.",
    "What is copy-on-write (CoW) in modern filesystems and fork()?",
    "Describe how HTTP/2 multiplexing solves head-of-line blocking at the app layer.",
    "How does QUIC use UDP to improve connection migration on mobile networks?",
    "Explain the role of attention masks in causal transformer decoding.",
    "What is quantization-aware training vs post-training weight quantization?",
    "Describe how continuous batching improves GPU utilization over static batching.",
    "Why does KV cache memory grow linearly with batch size and sequence length?",
    "Explain how speculative decoding uses a draft model to accelerate LLM inference.",
    "Compare tensor parallelism and pipeline parallelism for multi-GPU serving.",
    "How does PagedAttention eliminate external memory fragmentation in KV caches?",
    "Explain how RadixAttention stores prefix KV caches in a radix tree.",
    "What is FlashAttention and how does tiling reduce HBM memory bandwidth?",
    "Describe how rotary position embeddings (RoPE) encode relative token distances.",
    "How do Mixture-of-Experts (MoE) routers balance token loads across experts?",
    "Explain the trade-off between Time to First Token (TTFT) and Inter-Token Latency (ITL).",
]


def get_concurrent_prompts(count: int = 32) -> list[str]:
    """Return `count` distinct prompts for concurrent load testing."""
    if count <= len(CONCURRENT_PROMPT_POOL):
        return CONCURRENT_PROMPT_POOL[:count]
    result: list[str] = []
    for i in range(count):
        base = CONCURRENT_PROMPT_POOL[i % len(CONCURRENT_PROMPT_POOL)]
        result.append(f"[Request #{i + 1}] {base}")
    return result


AGENT_SYSTEM_PROMPT = """You are an expert Cloud Infrastructure & FinOps AI Agent equipped with a deterministic financial calculator tool.
Your mission is to help engineering teams model GPU cluster costs, concurrency capacity, reserved instance discounts, and per-user unit economics accurately.

### Available Tool Specification
```json
{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "Evaluates a mathematical expression for cloud infrastructure cost and capacity modeling.",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "Arithmetic expression using +, -, *, /, and parentheses, e.g. '8 * 2.50 * 720'"
        },
        "unit": {
          "type": "string",
          "description": "Unit of the output value, such as 'USD/month', 'tokens/sec', or 'USD/user'"
        }
      },
      "required": ["expression", "unit"]
    }
  }
}
```

### Operating Guidelines for Multi-Turn Reasoning
1. Always break down complex infrastructure sizing problems step by step across conversation turns.
2. When given a new cost or scaling parameter, state the formula clearly, show the exact arithmetic expression you would pass to `calculator`, and interpret the resulting figure in two concise sentences.
3. Preserve context from all previous turns (baseline monthly GPU spend, scaled cluster spend, reserved discount rate, and active concurrent user count).
4. Keep every response concise (under 90 words) and structured with bullet points so downstream automation can parse your reasoning.
5. Never invent numbers that contradict the tool output provided in the conversation history.
6. Reference earlier turn outputs explicitly when computing comparative savings or amortized unit metrics.
7. Format all currency values in USD with two decimal places and thousand separators so financial reports remain consistent across all five turns.
"""


def get_agent_turn_script(session_id: int = 0) -> list[dict[str, str]]:
    """Return a 5-turn agent interaction script with tool results injected between turns."""
    gpu_count = 8 + (session_id % 4) * 2
    hourly_rate = 2.50
    monthly_hours = 720
    turn1_cost = gpu_count * hourly_rate * monthly_hours
    turn2_cost = turn1_cost * 3.6
    turn3_cost = turn2_cost * 0.65
    active_users = 1200 + session_id * 100
    per_user_cost = round(turn3_cost / active_users, 2)

    return [
        {
            "user_message": (
                f"Session #{session_id}: We are deploying {gpu_count} NVIDIA T4 GPUs at "
                f"${hourly_rate:.2f}/hour for {monthly_hours} hours/month. "
                "Call the calculator concept and explain the baseline monthly GPU cost."
            ),
            "tool_result": (
                f"[Tool Output: calculator('{gpu_count} * {hourly_rate} * {monthly_hours}', 'USD/month') "
                f"-> {turn1_cost:,.2f} USD/month]"
            ),
        },
        {
            "user_message": (
                "Traffic grew 3.6x during peak product launch. Using the calculator result from Turn 1, "
                "what is our new projected monthly cluster spend before discounts?"
            ),
            "tool_result": (
                f"[Tool Output: calculator('{turn1_cost} * 3.6', 'USD/month') "
                f"-> {turn2_cost:,.2f} USD/month]"
            ),
        },
        {
            "user_message": (
                "Our cloud provider offered a 35% Reserved Instance discount (paying 65% of the peak price). "
                "Based on the Turn 2 tool output, compute the discounted monthly spend and monthly dollar savings."
            ),
            "tool_result": (
                f"[Tool Output: calculator('{turn2_cost} * 0.65', 'USD/month') "
                f"-> {turn3_cost:,.2f} USD/month]"
            ),
        },
        {
            "user_message": (
                f"Using the discounted monthly cost from Turn 3, if we serve {active_users} concurrent enterprise "
                "users, what is the amortized monthly GPU cost per concurrent user?"
            ),
            "tool_result": (
                f"[Tool Output: calculator('{turn3_cost} / {active_users}', 'USD/user') "
                f"-> {per_user_cost:,.2f} USD/user]"
            ),
        },
        {
            "user_message": (
                "Now produce a concise 4-line executive summary table combining Turn 1 baseline spend, "
                "Turn 2 peak spend, Turn 3 discounted spend, and Turn 4 cost per concurrent user."
            ),
        },
    ]


"""GPU memory monitoring helper using nvidia-smi with CPU-safe fallback."""


import subprocess


def query_gpu_memory_mb(device_index: int = 0) -> float:
    """Return used GPU memory in MiB for `device_index`, or 0.0 if unavailable."""
    cmd = [
        "nvidia-smi",
        f"--id={device_index}",
        "--query-gpu=memory.used",
        "--format=csv,noheader,nounits",
    ]
    try:
        raw = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=5)
        first_line = raw.decode("utf-8", errors="replace").strip().splitlines()[0]
        return float(first_line.strip())
    except (FileNotFoundError, subprocess.SubprocessError, ValueError, IndexError):
        return 0.0

"""Async OpenAI-compatible SSE streaming benchmark runner for vLLM and SGLang."""


import asyncio
import json
import math
import statistics
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from typing import Any



@dataclass
class RequestMetrics:
    """Metrics captured from a single streaming `/v1/chat/completions` call."""

    success: bool
    ttft_ms: float
    latency_s: float
    output_tokens: int
    response_text: str
    error: str = ""


def _percentile(sorted_values: list[float], pct: float) -> float:
    """Compute percentile (`0..100`) on a pre-sorted non-empty list."""
    if not sorted_values:
        return 0.0
    if len(sorted_values) == 1:
        return sorted_values[0]
    rank = (pct / 100.0) * (len(sorted_values) - 1)
    low = int(math.floor(rank))
    high = int(math.ceil(rank))
    if low == high:
        return sorted_values[low]
    weight = rank - low
    return sorted_values[low] * (1.0 - weight) + sorted_values[high] * weight


def _sync_stream_chat_completion(
    base_url: str,
    model: str,
    messages: list[dict[str, str]],
    max_tokens: int = 128,
    temperature: float = 0.0,
    timeout_s: float = 120.0,
) -> RequestMetrics:
    """Synchronous SSE streaming request executed in a worker thread."""
    url = f"{base_url.rstrip('/')}/v1/chat/completions"
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json", "Accept": "text/event-stream"},
        method="POST",
    )

    start_time = time.perf_counter()
    first_token_time: float | None = None
    chunks: list[str] = []
    content_chunk_count = 0
    reported_completion_tokens: int | None = None

    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            for raw_line in resp:
                line = raw_line.decode("utf-8", errors="replace").strip()
                if not line or line.startswith(":"):
                    continue
                if not line.startswith("data:"):
                    continue
                data_str = line[len("data:") :].strip()
                if data_str == "[DONE]":
                    break
                try:
                    event = json.loads(data_str)
                except json.JSONDecodeError:
                    continue

                usage = event.get("usage")
                if isinstance(usage, dict) and usage.get("completion_tokens") is not None:
                    reported_completion_tokens = int(usage["completion_tokens"])

                choices = event.get("choices") or []
                if choices:
                    delta = choices[0].get("delta") or {}
                    content = delta.get("content")
                    if content:
                        if first_token_time is None:
                            first_token_time = time.perf_counter()
                        chunks.append(content)
                        content_chunk_count += 1

        end_time = time.perf_counter()
        if not chunks or first_token_time is None:
            return RequestMetrics(
                success=False,
                ttft_ms=0.0,
                latency_s=0.0,
                output_tokens=0,
                response_text="",
                error="Empty response stream (no content chunks received)",
            )
        latency_s = max(end_time - start_time, 1e-6)
        ttft_ms = (first_token_time - start_time) * 1000.0
        full_text = "".join(chunks)
        output_tokens = (
            reported_completion_tokens
            if reported_completion_tokens is not None
            else max(content_chunk_count, len(full_text.split()))
        )
        return RequestMetrics(
            success=True,
            ttft_ms=ttft_ms,
            latency_s=latency_s,
            output_tokens=output_tokens,
            response_text=full_text,
        )
    except urllib.error.HTTPError as exc:
        return RequestMetrics(
            success=False,
            ttft_ms=0.0,
            latency_s=0.0,
            output_tokens=0,
            response_text="",
            error=f"HTTP {exc.code}: {exc.reason}",
        )
    except Exception as exc:
        return RequestMetrics(
            success=False,
            ttft_ms=0.0,
            latency_s=0.0,
            output_tokens=0,
            response_text="",
            error=str(exc),
        )


async def stream_chat_completion(
    base_url: str,
    model: str,
    messages: list[dict[str, str]],
    max_tokens: int = 128,
    temperature: float = 0.0,
    timeout_s: float = 120.0,
) -> RequestMetrics:
    """Async wrapper for streaming OpenAI chat completions."""
    return await asyncio.to_thread(
        _sync_stream_chat_completion,
        base_url,
        model,
        messages,
        max_tokens,
        temperature,
        timeout_s,
    )


def summarize_request_metrics(
    metrics: list[RequestMetrics], wall_time_s: float
) -> dict[str, Any]:
    """Aggregate a list of RequestMetrics into mean, P95, and throughput stats."""
    good = [m for m in metrics if m.success]
    failed_count = len(metrics) - len(good)
    if not good:
        return {
            "successful_requests": 0,
            "failed_requests": failed_count,
            "ttft_mean_ms": 0.0,
            "ttft_p95_ms": 0.0,
            "latency_mean_s": 0.0,
            "latency_p95_s": 0.0,
            "request_tok_per_s": 0.0,
            "total_output_tok_per_s": 0.0,
            "wall_time_s": round(wall_time_s, 4),
        }

    ttfts = sorted(m.ttft_ms for m in good)
    latencies = sorted(m.latency_s for m in good)
    req_speeds = [m.output_tokens / max(m.latency_s, 1e-6) for m in good]
    total_tokens = sum(m.output_tokens for m in good)

    return {
        "successful_requests": len(good),
        "failed_requests": failed_count,
        "ttft_mean_ms": round(statistics.mean(ttfts), 2),
        "ttft_p95_ms": round(_percentile(ttfts, 95.0), 2),
        "latency_mean_s": round(statistics.mean(latencies), 4),
        "latency_p95_s": round(_percentile(latencies, 95.0), 4),
        "request_tok_per_s": round(statistics.mean(req_speeds), 2),
        "total_output_tok_per_s": round(total_tokens / max(wall_time_s, 1e-6), 2),
        "wall_time_s": round(wall_time_s, 4),
    }


def _average_trial_summaries(trial_dicts: list[dict[str, Any]]) -> dict[str, Any]:
    """Average numeric metrics across multiple trials and compute standard deviations."""
    if not trial_dicts:
        return {}
    keys = [
        "ttft_mean_ms",
        "ttft_p95_ms",
        "latency_mean_s",
        "latency_p95_s",
        "request_tok_per_s",
        "total_output_tok_per_s",
        "wall_time_s",
    ]
    merged: dict[str, Any] = {
        "trials": len(trial_dicts),
        "successful_requests": sum(t["successful_requests"] for t in trial_dicts),
        "failed_requests": sum(t["failed_requests"] for t in trial_dicts),
    }
    for k in keys:
        vals = [float(t[k]) for t in trial_dicts]
        merged[k] = round(statistics.mean(vals), 3)
        merged[f"{k}_std"] = round(statistics.stdev(vals), 3) if len(vals) > 1 else 0.0
    return merged


async def run_warmup(base_url: str, model: str, warmup_requests: int = 3) -> None:
    """Send warmup requests to initialize CUDA graphs and JIT kernels."""
    for _ in range(warmup_requests):
        await stream_chat_completion(
            base_url=base_url,
            model=model,
            messages=[{"role": "user", "content": SINGLE_REQUEST_PROMPT}],
            max_tokens=32,
        )


async def run_single_request_benchmark(
    base_url: str,
    model: str,
    trials: int = 3,
    max_tokens: int = 128,
) -> dict[str, Any]:
    """Run Workload 1: Single request baseline repeated `trials` times."""
    trial_summaries: list[dict[str, Any]] = []
    sample_answer = ""
    for _ in range(trials):
        t0 = time.perf_counter()
        metric = await stream_chat_completion(
            base_url=base_url,
            model=model,
            messages=[{"role": "user", "content": SINGLE_REQUEST_PROMPT}],
            max_tokens=max_tokens,
        )
        wall = time.perf_counter() - t0
        if metric.response_text and not sample_answer:
            sample_answer = metric.response_text
        trial_summaries.append(summarize_request_metrics([metric], wall))

    res = _average_trial_summaries(trial_summaries)
    res["sample_response"] = sample_answer
    return res


async def run_concurrency_benchmark(
    base_url: str,
    model: str,
    concurrency_levels: tuple[int, ...] = (1, 4, 8, 16),
    num_requests: int = 32,
    trials: int = 3,
    max_tokens: int = 128,
) -> dict[str, dict[str, Any]]:
    """Run Workload 2: Concurrent requests across `concurrency_levels`."""
    prompts = get_concurrent_prompts(num_requests)
    results_by_level: dict[str, dict[str, Any]] = {}

    for conc in concurrency_levels:
        trial_summaries: list[dict[str, Any]] = []
        for _ in range(trials):
            sem = asyncio.Semaphore(conc)

            async def _worker(prompt_text: str) -> RequestMetrics:
                async with sem:
                    return await stream_chat_completion(
                        base_url=base_url,
                        model=model,
                        messages=[{"role": "user", "content": prompt_text}],
                        max_tokens=max_tokens,
                    )

            t0 = time.perf_counter()
            metrics = await asyncio.gather(*[_worker(p) for p in prompts])
            wall = time.perf_counter() - t0
            trial_summaries.append(summarize_request_metrics(list(metrics), wall))

        results_by_level[str(conc)] = _average_trial_summaries(trial_summaries)

    return results_by_level


async def _run_single_agent_session(
    base_url: str,
    model: str,
    session_id: int,
    max_tokens: int = 128,
) -> list[RequestMetrics]:
    """Execute one 5-turn agent session, growing the message history each turn."""
    history: list[dict[str, str]] = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT}
    ]
    script = get_agent_turn_script(session_id=session_id)
    turn_metrics: list[RequestMetrics] = []

    for turn in script:
        history.append({"role": "user", "content": turn["user_message"]})
        metric = await stream_chat_completion(
            base_url=base_url,
            model=model,
            messages=list(history),
            max_tokens=max_tokens,
        )
        turn_metrics.append(metric)
        assistant_reply = (
            metric.response_text
            if metric.success and metric.response_text
            else "Calculation acknowledged."
        )
        history.append({"role": "assistant", "content": assistant_reply})
        if "tool_result" in turn:
            history.append({"role": "user", "content": turn["tool_result"]})

    return turn_metrics


async def run_multiturn_agent_benchmark(
    base_url: str,
    model: str,
    parallel_sessions: tuple[int, ...] = (1, 4),
    trials: int = 3,
    max_tokens: int = 128,
) -> dict[str, dict[str, Any]]:
    """Run Workload 3: 5-turn tool-use agent conversation (single & multi-session)."""
    results_by_sessions: dict[str, dict[str, Any]] = {}

    for num_sessions in parallel_sessions:
        trial_summaries: list[dict[str, Any]] = []
        per_turn_ttfts_across_trials: list[list[float]] = [[] for _ in range(5)]

        for _ in range(trials):
            t0 = time.perf_counter()
            session_lists = await asyncio.gather(
                *[
                    _run_single_agent_session(
                        base_url=base_url,
                        model=model,
                        session_id=s,
                        max_tokens=max_tokens,
                    )
                    for s in range(num_sessions)
                ]
            )
            wall = time.perf_counter() - t0
            flat_metrics: list[RequestMetrics] = []
            for s_metrics in session_lists:
                flat_metrics.extend(s_metrics)
                for turn_idx, m in enumerate(s_metrics):
                    if m.success:
                        per_turn_ttfts_across_trials[turn_idx].append(m.ttft_ms)

            trial_summaries.append(summarize_request_metrics(flat_metrics, wall))

        agg = _average_trial_summaries(trial_summaries)
        agg["per_turn_ttft_ms"] = [
            round(statistics.mean(turn_vals), 2) if turn_vals else 0.0
            for turn_vals in per_turn_ttfts_across_trials
        ]
        results_by_sessions[str(num_sessions)] = agg

    return results_by_sessions


async def run_full_suite(
    engine_name: str,
    base_url: str,
    model: str = "Qwen/Qwen2.5-1.5B-Instruct",
    gpu_index: int = 0,
    trials: int = 3,
    num_concurrent_requests: int = 32,
    concurrency_levels: tuple[int, ...] = (1, 4, 8, 16),
    parallel_sessions: tuple[int, ...] = (1, 4),
    warmup_requests: int = 3,
    max_tokens: int = 128,
    baseline_gpu_mem_mb: float = 0.0,
) -> dict[str, Any]:
    """Execute Warmup + Workloads 1, 2, and 3 and return a structured JSON dictionary."""
    loaded_gpu_mem_mb = query_gpu_memory_mb(gpu_index)
    await run_warmup(base_url=base_url, model=model, warmup_requests=warmup_requests)

    single_res = await run_single_request_benchmark(
        base_url=base_url, model=model, trials=trials, max_tokens=max_tokens
    )
    peak_mem_after_single = query_gpu_memory_mb(gpu_index)

    conc_res = await run_concurrency_benchmark(
        base_url=base_url,
        model=model,
        concurrency_levels=concurrency_levels,
        num_requests=num_concurrent_requests,
        trials=trials,
        max_tokens=max_tokens,
    )
    peak_mem_after_conc = query_gpu_memory_mb(gpu_index)

    agent_res = await run_multiturn_agent_benchmark(
        base_url=base_url,
        model=model,
        parallel_sessions=parallel_sessions,
        trials=trials,
        max_tokens=max_tokens,
    )
    peak_mem_after_agent = query_gpu_memory_mb(gpu_index)

    peak_gpu_mem_mb = max(
        loaded_gpu_mem_mb,
        peak_mem_after_single,
        peak_mem_after_conc,
        peak_mem_after_agent,
    )

    return {
        "engine": engine_name,
        "model": model,
        "gpu_memory": {
            "baseline_mb": round(baseline_gpu_mem_mb, 1),
            "loaded_mb": round(loaded_gpu_mem_mb, 1),
            "peak_mb": round(peak_gpu_mem_mb, 1),
        },
        "single_request": single_res,
        "concurrency": conc_res,
        "multiturn_agent": agent_res,
    }

def format_single_engine_markdown(data: dict) -> str:
    """Format a single engine's benchmark results into ready-to-copy Markdown tables."""
    engine = data.get("engine", "unknown").upper()
    model = data.get("model", "Qwen/Qwen2.5-1.5B-Instruct")
    mem = data.get("gpu_memory", {})
    single = data.get("single_request", {})
    conc = data.get("concurrency", {})
    agent = data.get("multiturn_agent", {})

    lines = [
        f"# {engine} Benchmark Results (`{model}`)",
        "",
        "## 1. GPU Memory & Single-Request Baseline",
        "",
        "| Metric | Value |",
        "| :--- | ---: |",
        f"| **Baseline GPU Memory (MB)** | {mem.get('baseline_mb', 0.0):.1f} |",
        f"| **Loaded GPU Memory (MB)** | {mem.get('loaded_mb', 0.0):.1f} |",
        f"| **Peak GPU Memory (MB)** | {mem.get('peak_mb', 0.0):.1f} |",
        f"| **Single Request TTFT Mean (ms)** | {single.get('ttft_mean_ms', 0.0):.2f} ± {single.get('ttft_mean_ms_std', 0.0):.2f} |",
        f"| **Single Request TTFT P95 (ms)** | {single.get('ttft_p95_ms', 0.0):.2f} |",
        f"| **Single Request Total Latency (s)** | {single.get('latency_mean_s', 0.0):.3f} ± {single.get('latency_mean_s_std', 0.0):.3f} |",
        f"| **Single Request Output Speed (tok/s)** | {single.get('request_tok_per_s', 0.0):.2f} |",
        f"| **Failed Requests** | {single.get('failed_requests', 0)} |",
        "",
        f"**Sample Response:** _{single.get('sample_response', '')}_",
        "",
        "## 2. Concurrent Users Benchmark (32 Prompts per Batch)",
        "",
        "| Concurrency | Mean TTFT (ms) | P95 TTFT (ms) | Mean Latency (s) | Per-Req Speed (tok/s) | Total Server Throughput (tok/s) | Failed |",
        "| :---: | ---: | ---: | ---: | ---: | ---: | :---: |",
    ]

    for lvl in sorted(conc.keys(), key=lambda x: int(x)):
        c = conc[lvl]
        lines.append(
            f"| **{lvl} users** | {c.get('ttft_mean_ms', 0.0):.1f} | {c.get('ttft_p95_ms', 0.0):.1f} | "
            f"{c.get('latency_mean_s', 0.0):.3f} | {c.get('request_tok_per_s', 0.0):.1f} | "
            f"**{c.get('total_output_tok_per_s', 0.0):.1f}** | {c.get('failed_requests', 0)} |"
        )

    lines.extend([
        "",
        "## 3. Multi-Turn Agent Benchmark (5 Turns + Calculator Tool)",
        "",
        "| Parallel Agent Sessions | Mean TTFT (ms) | P95 TTFT (ms) | Mean Latency (s) | Total Throughput (tok/s) | Turn 1 -> Turn 5 TTFT (ms) | Failed |",
        "| :---: | ---: | ---: | ---: | ---: | :--- | :---: |",
    ])

    for sess in sorted(agent.keys(), key=lambda x: int(x)):
        a = agent[sess]
        turns = a.get("per_turn_ttft_ms", [])
        t_str = " -> ".join(f"{x:.1f}" for x in turns) if turns else "N/A"
        lines.append(
            f"| **{sess} session(s)** | {a.get('ttft_mean_ms', 0.0):.1f} | {a.get('ttft_p95_ms', 0.0):.1f} | "
            f"{a.get('latency_mean_s', 0.0):.3f} | **{a.get('total_output_tok_per_s', 0.0):.1f}** | "
            f"`{t_str}` | {a.get('failed_requests', 0)} |"
        )

    return "\n".join(lines) + "\n"


In [ ]:
# Launch command: python3 -m sglang.launch_server --model-path Qwen/Qwen2.5-1.5B-Instruct --dtype half --context-length 4096 --mem-fraction-static 0.85 --attention-backend triton --port 30000
import os
import subprocess
import time
import urllib.request

BASELINE_GPU_MEM_MB = query_gpu_memory_mb(0)
print(f"Baseline GPU 0 memory before starting sglang: {BASELINE_GPU_MEM_MB:.1f} MB")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
cmd = ["python3", "-m", "sglang.launch_server", "--model-path", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--context-length", "4096", "--mem-fraction-static", "0.85", "--attention-backend", "triton", "--port", "30000"]
print("Launching:", " ".join(cmd))

log_file = open("sglang_server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env)

base_url = "http://127.0.0.1:30000"
ready = False
for attempt in range(120):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"{base_url}/v1/models", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                print(f"Server ready after {attempt * 2}s!")
                break
    except Exception:
        time.sleep(2)

if not ready:
    log_file.flush()
    print("Server failed to start. Last 50 log lines:")
    print("\n".join(open("sglang_server.log").read().splitlines()[-50:]))
    raise RuntimeError("sglang server failed to start")


In [ ]:
import json
from IPython.display import Markdown, display

results = await run_full_suite(
    engine_name="sglang",
    base_url="http://127.0.0.1:30000",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_index=0,
    trials=3,
    num_concurrent_requests=32,
    concurrency_levels=(1, 4, 8, 16),
    parallel_sessions=(1, 4),
    warmup_requests=3,
    max_tokens=128,
    baseline_gpu_mem_mb=BASELINE_GPU_MEM_MB,
)

with open("sglang_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

summary_md = format_single_engine_markdown(results)
with open("sglang_summary.md", "w", encoding="utf-8") as f:
    f.write(summary_md)

print("Saved benchmark results to sglang_results.json and sglang_summary.md\n")
print(summary_md)
display(Markdown(summary_md))


In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print("Server stopped cleanly and GPU memory released.")
if 'log_file' in globals() and not log_file.closed:
    log_file.close()
